# PechayDetect — Model A: MobileNetV2

**Thesis:** Detection of Nutrient Deficiencies in Pechay (*Brassica rapa* subsp. *chinensis*) Using Deep Learning

**Architecture:** MobileNetV2 (Transfer Learning + Advanced Fine-Tuning)

---

## Python Version Justification

This notebook uses **Python 3.10**. Rationale:

- TensorFlow 2.13 officially supports Python 3.8, 3.9, 3.10, and 3.11.
- Python 3.12 is **not** officially supported by TensorFlow 2.x (as of TF 2.15).
- Python 3.10 is the **default runtime** in Google Colab (as of 2024), ensuring zero version conflicts.
- Both `tf.keras.applications.MobileNetV2` and `tf.keras.applications.MobileNetV3Large` are stable under Python 3.10 + TF 2.13.

Source: [TensorFlow install guide](https://www.tensorflow.org/install/pip#software_requirements)

---

## Advanced Training Strategy

This notebook implements the following advanced techniques:

1. **Mixed Precision Training** (float16) — speeds up GPU training with minimal accuracy loss
2. **Three-Phase Progressive Fine-Tuning** — head-only → top layers → full model
3. **Cosine Decay LR Schedule with Linear Warmup** — stable convergence
4. **Label Smoothing** — reduces overconfidence, improves calibration
5. **Class Weighting** — handles class imbalance automatically
6. **Keras Preprocessing Augmentation** — baked into the data pipeline
7. **Test-Time Augmentation (TTA)** — improves inference accuracy
8. **Grad-CAM** — visual explanations for model decisions
9. **Model Calibration** — reliability diagram + ECE
10. **OOD Rejection** — confidence threshold + entropy-based analysis
11. **Three TFLite Formats** — float32, float16, int8 with calibration


## Section 1: Environment Setup

Install required packages and verify versions. Run this cell once per Colab session.

In [ ]:
# Install pinned versions for reproducibility
# TensorFlow 2.13 is used: stable, supports MobileNetV2/V3, Python 3.10, mixed precision
!pip install -q tensorflow==2.13.0
!pip install -q scikit-learn==1.3.2
!pip install -q matplotlib seaborn pandas numpy pillow tqdm

import os
import sys
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF info/warnings

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras import mixed_precision
from tensorflow.keras.applications import MobileNetV2

import sklearn
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

print(f'Python:      {sys.version}')
print(f'TensorFlow:  {tf.__version__}')
print(f'Keras:       {keras.__version__}')
print(f'scikit-learn:{sklearn.__version__}')
print(f'NumPy:       {np.__version__}')


In [ ]:
# --- GPU detection and mixed precision setup ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {gpus}')
    # Enable mixed precision (float16 compute, float32 storage)
    # Reduces memory usage and speeds up training on NVIDIA T4/V100 (Colab GPUs)
    # The final softmax is kept in float32 via explicit dtype='float32' on that layer
    mixed_precision.set_global_policy('mixed_float16')
    print(f'Mixed precision enabled: {mixed_precision.global_policy()}')
else:
    print('No GPU detected — training on CPU (slower). Consider enabling GPU in Colab:')
    print('Runtime → Change runtime type → Hardware accelerator → GPU')
    mixed_precision.set_global_policy('float32')


## Section 2: Configuration

All configurable parameters are defined here. Modify this cell to adjust hyperparameters.

In [ ]:
# ================================================================
# CONFIGURATION — Edit this section to match your setup
# ================================================================

# --- Reproducibility seed ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Dataset paths ---
# Option 1: Local path inside Colab VM
DATASET_ROOT = '/content/Dataset'

# Option 2: Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_ROOT = '/content/drive/MyDrive/PechayDetect/Dataset'

TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
VAL_DIR   = os.path.join(DATASET_ROOT, 'validation')
TEST_DIR  = os.path.join(DATASET_ROOT, 'test')

# Output directory for checkpoints, logs, and converted models
OUTPUT_DIR = '/content/outputs/mobilenetv2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'logs'), exist_ok=True)

# --- Dataset split documentation ---
# These values document the intended split ratio.
# Actual splitting is done by your pre-split folder structure.
# If your data is in a single folder, use image_dataset_from_directory
# with validation_split=VALIDATION_SPLIT and subset='training'/'validation'.
TRAIN_SPLIT      = 0.80
VALIDATION_SPLIT = 0.20

# --- Image parameters ---
# MobileNetV2 standard input: 224x224x3 (from original paper)
# Howard et al. (2017). MobileNets: Efficient Convolutional Neural Networks
# for Mobile Vision Applications. arXiv:1704.04861
IMAGE_SIZE  = (224, 224)
CHANNELS    = 3
INPUT_SHAPE = IMAGE_SIZE + (CHANNELS,)

# --- Class names ---
# Order must match your folder names exactly.
# Add 'Unknown' folder to TRAIN_DIR/VAL_DIR/TEST_DIR to enable 5-class OOD training.
CLASS_NAMES = ['Healthy', 'Nitrogen', 'Phosphorus', 'Potassium']
# CLASS_NAMES = ['Healthy', 'Nitrogen', 'Phosphorus', 'Potassium', 'Unknown']  # 5-class OOD
NUM_CLASSES = len(CLASS_NAMES)

# --- Training hyperparameters ---
BATCH_SIZE    = 32
EPOCHS_PHASE1 = 20   # Phase 1: Frozen base, train head only
EPOCHS_PHASE2 = 20   # Phase 2: Unfreeze top layers
EPOCHS_PHASE3 = 15   # Phase 3: Unfreeze all layers

# --- Learning rates ---
LR_PHASE1 = 1e-3   # High LR for new head
LR_PHASE2 = 1e-4   # Lower for partial fine-tuning
LR_PHASE3 = 1e-5   # Very low for full fine-tuning

# --- MobileNetV2 fine-tuning ---
# MobileNetV2 (alpha=1.0) has 154 layers.
# Phase 2 unfreezes layers from FINE_TUNE_FROM onward.
FINE_TUNE_FROM_PHASE2 = 100  # Unfreeze last ~54 layers
FINE_TUNE_FROM_PHASE3 = 0    # Unfreeze all layers

# --- Regularization ---
DROPOUT_RATE      = 0.3
L2_REGULARIZATION = 1e-4
LABEL_SMOOTHING   = 0.1   # Prevents overconfidence; improves calibration

# --- OOD rejection threshold ---
# Predictions with max(softmax) < CONFIDENCE_THRESHOLD are rejected as 'Unknown'.
# Tune this value based on your validation set analysis.
CONFIDENCE_THRESHOLD = 0.70

# --- Paths ---
MODEL_NAME       = 'mobilenetv2_pechaydetect'
CHECKPOINT_PATH  = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_best.h5')
SAVEDMODEL_PATH  = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_savedmodel')
TFLITE_PATH_F32  = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_float32.tflite')
TFLITE_PATH_F16  = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_float16.tflite')
TFLITE_PATH_INT8 = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_int8.tflite')

print('Configuration loaded.')
print(f'  Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'  Image size:     {IMAGE_SIZE}')
print(f'  Batch size:     {BATCH_SIZE}')
print(f'  Output dir:     {OUTPUT_DIR}')


## Section 3: Dataset Verification and Loading

**Expected folder structure:**
```
Dataset/
  train/
    Healthy/       ← training images for Healthy class
    Nitrogen/
    Phosphorus/
    Potassium/
    Unknown/       ← optional: non-Pechay images for OOD training
  validation/
    Healthy/
    Nitrogen/
    Phosphorus/
    Potassium/
    Unknown/       ← optional
  test/
    Healthy/
    Nitrogen/
    Phosphorus/
    Potassium/
    Unknown/       ← optional
```

**The test set is never used during training or validation.**
It is used only for the final evaluation in Section 10.

In [ ]:
def verify_dataset_structure(split_dirs, class_names):
    """
    Verify dataset directory structure and report image counts.

    Args:
        split_dirs: dict mapping split name to path
        class_names: list of expected class folder names

    Raises:
        FileNotFoundError: if required directories are missing
    """
    IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    issues = []
    summary = {}

    for split_name, dir_path in split_dirs.items():
        summary[split_name] = {}
        if not os.path.isdir(dir_path):
            issues.append(f'Missing directory: {dir_path}')
            continue
        for cls in class_names:
            cls_path = os.path.join(dir_path, cls)
            if not os.path.isdir(cls_path):
                issues.append(f'Missing class folder: {cls_path}')
                summary[split_name][cls] = 0
            else:
                n = sum(1 for f in os.listdir(cls_path)
                        if os.path.splitext(f)[1].lower() in IMAGE_EXTS)
                summary[split_name][cls] = n
                if n == 0:
                    issues.append(f'No images found in: {cls_path}')

    # Print summary table
    header = f"{'Class':<15}" + ''.join(f"{s:>12}" for s in split_dirs)
    print(header)
    print('-' * len(header))
    for cls in class_names:
        row = f"{cls:<15}"
        for split_name in split_dirs:
            row += f"{summary[split_name].get(cls, 0):>12}"
        print(row)
    total_row = f"{'TOTAL':<15}"
    for split_name in split_dirs:
        total_row += f"{sum(summary[split_name].values()):>12}"
    print('-' * len(header))
    print(total_row)

    if issues:
        print('\n\u26a0\ufe0f  Issues found:')
        for issue in issues:
            print(f'  - {issue}')
        raise FileNotFoundError('Resolve the above dataset issues before continuing.')
    else:
        print('\n\u2705 Dataset structure verified.')
    return summary


print('Verifying dataset structure...')
split_dirs = {'Train': TRAIN_DIR, 'Validation': VAL_DIR, 'Test': TEST_DIR}
dataset_summary = verify_dataset_structure(split_dirs, CLASS_NAMES)


In [ ]:
# --- Class distribution bar chart ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4CAF50', '#F44336', '#2196F3', '#FF9800', '#9C27B0'][:NUM_CLASSES]

for ax, (split_name, counts) in zip(axes, dataset_summary.items()):
    values = [counts.get(c, 0) for c in CLASS_NAMES]
    bars = ax.bar(CLASS_NAMES, values, color=colors, edgecolor='white')
    ax.set_title(f'{split_name} Set', fontsize=13, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of Images')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, max(values) * 1.2 if max(values) > 0 else 10)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values) * 0.02,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('PechayDetect — Dataset Class Distribution (MobileNetV2)', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


## Section 4: Data Augmentation and Preprocessing

**Preprocessing:** MobileNetV2 was pretrained with pixel values in **[-1, 1]**.
We apply the formula: `pixel = (pixel / 127.5) - 1.0`.
This is identical to `tf.keras.applications.mobilenet_v2.preprocess_input`.

**Augmentation justification:**
| Augmentation | Justification |
|---|---|
| RandomFlip (H+V) | Leaves can be oriented in any direction in field photos |
| RandomRotation ±15° | Camera may be tilted slightly during capture |
| RandomZoom ±10% | Varying distances from leaf during capture |
| RandomBrightness ±20% | Outdoor lighting varies significantly |
| RandomContrast ±10% | Different camera exposures in the field |

Augmentation is applied **only to the training set**. Validation and test sets use only normalization.

In [ ]:
# --- Augmentation pipeline (applied only during training) ---
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    layers.RandomRotation(factor=0.15, fill_mode='reflect', seed=SEED),  # \u00b115\u00b0
    layers.RandomZoom(height_factor=(-0.1, 0.1), seed=SEED),             # \u00b110% zoom
    layers.RandomBrightness(factor=0.2, seed=SEED),                      # \u00b120% brightness
    layers.RandomContrast(factor=0.1, seed=SEED),                        # \u00b110% contrast
], name='data_augmentation')


def preprocess_mobilenetv2(image, label):
    """
    Normalize image to [-1, 1] as required by MobileNetV2's pretrained weights.
    Equivalent to tf.keras.applications.mobilenet_v2.preprocess_input.
    """
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0
    return image, label


def build_dataset(directory, split_name, augment=False):
    """
    Load and pipeline a dataset split.

    Args:
        directory (str): Path to the dataset split directory.
        split_name (str): One of 'train', 'validation', 'test'. Used for shuffle logic.
        augment (bool): Apply data augmentation if True.

    Returns:
        tf.data.Dataset: Optimized, batched, prefetched dataset.
    """
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',           # Class labels inferred from subdirectory names
        label_mode='categorical',    # One-hot encoded labels (shape: [batch, NUM_CLASSES])
        class_names=CLASS_NAMES,     # Enforce consistent class ordering
        color_mode='rgb',
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,       # Bilinear resizing to 224x224
        shuffle=(split_name == 'train'),
        seed=SEED,
        interpolation='bilinear',
        crop_to_aspect_ratio=False,  # Stretch to 224x224; consistent with paper implementations
    )

    # Apply MobileNetV2 normalization to [-1, 1]
    ds = ds.map(preprocess_mobilenetv2, num_parallel_calls=tf.data.AUTOTUNE)

    # Apply augmentation to training set only
    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # Cache after augmentation; prefetch to overlap compute and I/O
    ds = ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds


print('Loading datasets...')
train_ds = build_dataset(TRAIN_DIR, 'train',      augment=True)
val_ds   = build_dataset(VAL_DIR,   'validation', augment=False)
test_ds  = build_dataset(TEST_DIR,  'test',       augment=False)

print(f'  Train batches:      {train_ds.cardinality().numpy()}')
print(f'  Validation batches: {val_ds.cardinality().numpy()}')
print(f'  Test batches:       {test_ds.cardinality().numpy()}')
print('\u2705 Datasets loaded.')


In [ ]:
# --- Visualize sample training images (with augmentation) ---
def show_samples(dataset, class_names, n_cols=4, n_rows=2, title='Sample Images'):
    """Display images from the dataset. Denormalize from [-1,1] for display."""
    images, labels = next(iter(dataset))
    img_display = np.clip((images.numpy() + 1.0) / 2.0, 0, 1)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(img_display):
            ax.imshow(img_display[i])
            ax.set_title(class_names[np.argmax(labels[i])], fontsize=10)
        ax.axis('off')
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, title.lower().replace(' ', '_') + '.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


show_samples(train_ds, CLASS_NAMES, title='Train Samples with Augmentation')
show_samples(val_ds,   CLASS_NAMES, title='Validation Samples (No Augmentation)')


## Section 5: Class Weighting

Class weights compensate for imbalanced class distributions. If the dataset is balanced, all weights will be ~1.0 and training is unaffected.

In [ ]:
# --- Compute class weights from training set distribution ---
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def get_label_array(directory, class_names):
    """Build flat label array for class weight computation."""
    labels = []
    for idx, cls in enumerate(class_names):
        cls_path = os.path.join(directory, cls)
        if not os.path.isdir(cls_path):
            continue
        n = sum(1 for f in os.listdir(cls_path)
                if os.path.splitext(f)[1].lower() in IMAGE_EXTS)
        labels.extend([idx] * n)
    return np.array(labels)


train_labels = get_label_array(TRAIN_DIR, CLASS_NAMES)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)
class_weight_dict = {i: float(w) for i, w in enumerate(class_weights_arr)}

print('Class weights (balanced):')
for cls, w in zip(CLASS_NAMES, class_weights_arr):
    print(f'  {cls:<15}: {w:.4f}')


## Section 6: Learning Rate Schedule

**Cosine Decay with Linear Warmup:**

- **Warmup phase:** LR increases linearly from 0 to `initial_lr` over the first `warmup_steps`. This prevents instability when the randomly initialized head drives large gradients into the pretrained base.
- **Cosine decay phase:** LR decreases smoothly following a cosine curve, allowing gradual convergence without premature stopping.

Reference: Loshchilov & Hutter (2017). *SGDR: Stochastic Gradient Descent with Warm Restarts*. ICLR 2017. arXiv:1608.03983

In [ ]:
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    """
    Learning rate schedule: linear warmup followed by cosine decay.

    Args:
        initial_lr (float): Peak learning rate after warmup.
        warmup_steps (int): Number of steps for linear warmup.
        total_steps (int): Total number of training steps.
        min_lr (float): Minimum LR at end of cosine decay (default 0).
    """

    def __init__(self, initial_lr, warmup_steps, total_steps, min_lr=0.0):
        super().__init__()
        self.initial_lr   = float(initial_lr)
        self.warmup_steps = int(warmup_steps)
        self.total_steps  = int(total_steps)
        self.min_lr       = float(min_lr)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup = tf.cast(self.warmup_steps, tf.float32)
        total  = tf.cast(self.total_steps,  tf.float32)

        # Linear warmup
        warmup_lr = self.initial_lr * (step / tf.maximum(warmup, 1.0))

        # Cosine decay
        progress  = (step - warmup) / tf.maximum(total - warmup, 1.0)
        cosine_lr = self.min_lr + 0.5 * (self.initial_lr - self.min_lr) * (
            1.0 + tf.cos(math.pi * progress)
        )

        return tf.where(step < warmup, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'initial_lr':   self.initial_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps':  self.total_steps,
            'min_lr':       self.min_lr,
        }


def make_lr_schedule(initial_lr, n_epochs, steps_per_epoch, warmup_fraction=0.1):
    """Convenience factory for WarmupCosineDecay."""
    total   = n_epochs * steps_per_epoch
    warmup  = max(1, int(warmup_fraction * total))
    return WarmupCosineDecay(initial_lr, warmup, total)


STEPS_PER_EPOCH = len(train_ds)
print(f'Steps per epoch: {STEPS_PER_EPOCH}')

# Plot the LR schedule for Phase 1 as a sanity check
sched = make_lr_schedule(LR_PHASE1, EPOCHS_PHASE1, STEPS_PER_EPOCH)
lr_vals = [sched(i).numpy() for i in range(EPOCHS_PHASE1 * STEPS_PER_EPOCH)]
plt.figure(figsize=(10, 3))
plt.plot(lr_vals, linewidth=2)
plt.title('Phase 1 — Cosine Decay with Warmup LR Schedule')
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lr_schedule_phase1.png'), dpi=150, bbox_inches='tight')
plt.show()


## Section 7: Model Architecture

**MobileNetV2** (Sandler et al., 2018 — arXiv:1801.04381):
- Inverted residual blocks with linear bottlenecks
- Depthwise separable convolutions for efficiency
- Alpha=1.0 (full width, maximum accuracy)
- Pretrained on ImageNet (1.28M images, 1000 classes)

**Classification Head:**
1. `GlobalAveragePooling2D` — collapses spatial dimensions, retains channel info
2. `BatchNormalization` — stabilizes training; mild regularization
3. `Dense(256, ReLU, L2)` — task-specific feature transformation
4. `Dropout(0.3)` — stochastic regularization to prevent overfitting
5. `Dense(NUM_CLASSES)` + `Softmax (float32)` — output probabilities

Note: The final Softmax is explicitly cast to `float32` to avoid numerical issues with `mixed_float16` policy.

In [ ]:
def build_mobilenetv2_model(num_classes, input_shape, dropout_rate, l2_reg):
    """
    Build MobileNetV2 transfer learning model.

    Args:
        num_classes (int): Number of output classes.
        input_shape (tuple): Input shape (H, W, C).
        dropout_rate (float): Dropout probability in the head.
        l2_reg (float): L2 regularization coefficient.

    Returns:
        model (keras.Model): Full model.
        base_model (keras.Model): The MobileNetV2 backbone (for fine-tuning control).
    """
    # Load MobileNetV2 pretrained on ImageNet; exclude the classification head
    base_model = MobileNetV2(
        input_shape=input_shape,
        alpha=1.0,          # Width multiplier: 1.0 = full capacity
        include_top=False,  # Remove ImageNet head
        weights='imagenet', # Use pretrained ImageNet weights
        pooling=None,       # Pooling will be added manually
    )
    base_model.trainable = False  # Freeze all base layers initially (Phase 1)

    # Build functional model
    inputs = keras.Input(shape=input_shape, name='input_image')

    # Pass training=False so BN layers run in inference mode even during training
    # This is critical while the base is frozen (prevents BN statistics from shifting)
    x = base_model(inputs, training=False)

    # --- Classification head ---
    x = layers.GlobalAveragePooling2D(name='head_gap')(x)
    x = layers.BatchNormalization(name='head_bn')(x)
    x = layers.Dense(
        256,
        activation='relu',
        kernel_regularizer=keras.regularizers.l2(l2_reg),
        name='head_dense'
    )(x)
    x = layers.Dropout(dropout_rate, name='head_dropout')(x)

    # Linear logits layer (softmax applied separately in float32 for mixed precision safety)
    x = layers.Dense(num_classes, name='logits')(x)

    # Cast to float32 before softmax — required for numerical stability with float16 policy
    outputs = layers.Activation('softmax', dtype='float32', name='output_softmax')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='MobileNetV2_PechayDetect')
    return model, base_model


model, base_model = build_mobilenetv2_model(
    num_classes=NUM_CLASSES,
    input_shape=INPUT_SHAPE,
    dropout_rate=DROPOUT_RATE,
    l2_reg=L2_REGULARIZATION,
)

model.summary()
print(f'\nTotal layers in model:     {len(model.layers)}')
print(f'Total layers in backbone:  {len(base_model.layers)}')
trainable   = sum(v.numpy().size for v in model.trainable_variables)
untrainable = sum(v.numpy().size for v in model.non_trainable_variables)
print(f'Trainable params:   {trainable:,}')
print(f'Non-trainable params: {untrainable:,}')


## Section 8: Phase 1 Training — Classification Head Only

The base MobileNetV2 is **fully frozen**. Only the classification head is trained.

**Goal:** Learn task-specific features in the head without disrupting pretrained weights.

**Duration:** Up to `EPOCHS_PHASE1` epochs with early stopping.

In [ ]:
def get_standard_callbacks(phase_name, output_dir, checkpoint_path, monitor_metric='val_accuracy',
                           patience_stop=8, patience_reduce=4):
    """Return standard callback list for a training phase."""
    return [
        # Save model weights whenever validation accuracy improves
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor_metric,
            mode='max' if 'accuracy' in monitor_metric else 'min',
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        ),
        # Stop training when metric stops improving
        keras.callbacks.EarlyStopping(
            monitor=monitor_metric,
            mode='max' if 'accuracy' in monitor_metric else 'min',
            patience=patience_stop,
            restore_best_weights=True,
            verbose=1,
        ),
        # Log metrics to CSV for later analysis
        keras.callbacks.CSVLogger(
            filename=os.path.join(output_dir, f'{phase_name}_history.csv'),
            append=False,
        ),
        # TensorBoard logging (optional: run %tensorboard --logdir /content/outputs)
        keras.callbacks.TensorBoard(
            log_dir=os.path.join(output_dir, 'logs', phase_name),
            histogram_freq=1,
            write_graph=True,
        ),
    ]


# --- Compile model for Phase 1 ---
phase1_schedule = make_lr_schedule(LR_PHASE1, EPOCHS_PHASE1, STEPS_PER_EPOCH)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=phase1_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy'),
        keras.metrics.AUC(name='auc', multi_label=False),
    ],
)

print('=' * 60)
print('PHASE 1: Training classification head (base frozen)')
print('=' * 60)

history_p1 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE1,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=get_standard_callbacks('phase1', OUTPUT_DIR, CHECKPOINT_PATH),
    verbose=1,
)

print(f'\nPhase 1 best val_accuracy: {max(history_p1.history["val_accuracy"]):.4f}')


In [ ]:
def plot_phase_history(history, phase_label, output_dir):
    """Plot and save training/validation loss and accuracy curves for one phase."""
    h = history.history
    epochs = range(1, len(h['loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, h['loss'],     'b-o', markersize=4, label='Train')
    axes[0].plot(epochs, h['val_loss'], 'r-o', markersize=4, label='Validation')
    axes[0].set_title(f'{phase_label} — Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, h['accuracy'],     'b-o', markersize=4, label='Train')
    axes[1].plot(epochs, h['val_accuracy'], 'r-o', markersize=4, label='Validation')
    axes[1].set_title(f'{phase_label} — Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1.05])

    plt.suptitle(f'MobileNetV2 — {phase_label}', fontsize=14)
    plt.tight_layout()
    fname = os.path.join(output_dir, phase_label.lower().replace(' ', '_') + '_curves.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')


plot_phase_history(history_p1, 'Phase 1 (Frozen Base)', OUTPUT_DIR)


## Section 9: Phase 2 — Fine-Tune Top Layers

Unfreeze the **top portion** of MobileNetV2 (layers from `FINE_TUNE_FROM_PHASE2` onward).

**Key rule:** BatchNormalization layers in the base model remain frozen during fine-tuning to preserve learned mean/variance statistics from ImageNet pretraining. Retraining BN with a small dataset can destabilize the model.

Reference: Chollet (2021). *Deep Learning with Python, 2nd ed.* Manning Publications.

In [ ]:
def set_base_trainability(base_model, from_layer_idx):
    """
    Unfreeze base model layers from `from_layer_idx` onward.
    BatchNormalization layers are always kept frozen to preserve pretrained statistics.

    Args:
        base_model: The backbone Keras Model.
        from_layer_idx (int): Freeze layers before this index; unfreeze from here.
                              Use 0 to unfreeze all non-BN layers.
    """
    base_model.trainable = True
    for i, layer in enumerate(base_model.layers):
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False  # Always frozen
        elif i < from_layer_idx:
            layer.trainable = False  # Frozen (early base layers)
        else:
            layer.trainable = True   # Trainable

    trainable_n = sum(1 for l in base_model.layers if l.trainable
                      and not isinstance(l, layers.BatchNormalization))
    print(f'Base model trainable (non-BN) layers: {trainable_n}/{len(base_model.layers)}')


print('=' * 60)
print('PHASE 2: Fine-tuning top MobileNetV2 layers')
print(f'Unfreezing from layer {FINE_TUNE_FROM_PHASE2} of {len(base_model.layers)}')
print('=' * 60)

set_base_trainability(base_model, FINE_TUNE_FROM_PHASE2)

phase2_schedule = make_lr_schedule(LR_PHASE2, EPOCHS_PHASE2, STEPS_PER_EPOCH, warmup_fraction=0.05)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=phase2_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy'),
        keras.metrics.AUC(name='auc', multi_label=False),
    ],
)

history_p2 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE2,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=get_standard_callbacks('phase2', OUTPUT_DIR, CHECKPOINT_PATH, patience_stop=7),
    verbose=1,
)

print(f'\nPhase 2 best val_accuracy: {max(history_p2.history["val_accuracy"]):.4f}')
plot_phase_history(history_p2, 'Phase 2 (Top Layers Fine-Tuned)', OUTPUT_DIR)


## Section 10: Phase 3 — Full Model Fine-Tuning

Unfreeze **all non-BN** layers for final convergence with a very low learning rate.

**Risk:** Training all layers with too high a learning rate can destroy pretrained features ("catastrophic forgetting"). The very low LR (`1e-5`) and cosine schedule mitigate this.

In [ ]:
print('=' * 60)
print('PHASE 3: Full model fine-tuning (all non-BN layers)')
print('=' * 60)

set_base_trainability(base_model, from_layer_idx=0)  # Unfreeze all non-BN layers

phase3_schedule = make_lr_schedule(LR_PHASE3, EPOCHS_PHASE3, STEPS_PER_EPOCH, warmup_fraction=0.05)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=phase3_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy'),
        keras.metrics.AUC(name='auc', multi_label=False),
    ],
)

history_p3 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE3,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=get_standard_callbacks('phase3', OUTPUT_DIR, CHECKPOINT_PATH, patience_stop=5),
    verbose=1,
)

print(f'\nPhase 3 best val_accuracy: {max(history_p3.history["val_accuracy"]):.4f}')
plot_phase_history(history_p3, 'Phase 3 (Full Fine-Tuning)', OUTPUT_DIR)


In [ ]:
# --- Combined training history across all three phases ---
def plot_combined_history(h1, h2, h3, output_dir, model_name):
    """Concatenate phase histories and plot with phase boundaries marked."""
    acc     = h1.history['accuracy']     + h2.history['accuracy']     + h3.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy'] + h3.history['val_accuracy']
    loss    = h1.history['loss']         + h2.history['loss']         + h3.history['loss']
    val_loss= h1.history['val_loss']     + h2.history['val_loss']     + h3.history['val_loss']

    ep1_end = len(h1.history['accuracy'])
    ep2_end = ep1_end + len(h2.history['accuracy'])
    total   = len(acc)
    epochs  = range(1, total + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, (train_v, val_v, ylabel) in zip(axes, [
        (acc, val_acc, 'Accuracy'),
        (loss, val_loss, 'Loss'),
    ]):
        ax.plot(epochs, train_v, 'b-', lw=2, label='Train')
        ax.plot(epochs, val_v,   'r-', lw=2, label='Validation')
        ax.axvline(ep1_end + 0.5, color='gray',  ls='--', lw=1.5, label='Phase 2 start')
        ax.axvline(ep2_end + 0.5, color='green', ls='--', lw=1.5, label='Phase 3 start')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(f'{model_name} Combined — {ylabel}')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.suptitle(f'{model_name} — Full Training History (All Phases)', fontsize=14)
    plt.tight_layout()
    fname = os.path.join(output_dir, 'combined_training_history.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')


plot_combined_history(history_p1, history_p2, history_p3, OUTPUT_DIR, 'MobileNetV2')


## Section 11: Evaluation on Test Set

The test set is used **only here**, after all training is complete.
The best checkpoint (highest validation accuracy) is loaded for evaluation.

In [ ]:
print('=' * 60)
print('EVALUATION: Loading best checkpoint')
print('=' * 60)

best_model = keras.models.load_model(CHECKPOINT_PATH)
print(f'Loaded: {CHECKPOINT_PATH}')

# Keras-level evaluation
eval_results = best_model.evaluate(test_ds, verbose=1)
eval_dict = dict(zip(best_model.metrics_names, eval_results))
print('\nKeras evaluation:')
for name, val in eval_dict.items():
    print(f'  {name}: {val:.4f}')

# Collect all predictions and ground-truth labels
all_probs  = []
all_true   = []

for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    all_probs.extend(probs)
    all_true.extend(np.argmax(labels.numpy(), axis=1))

all_probs = np.array(all_probs)     # shape: (N, NUM_CLASSES)
all_true  = np.array(all_true)      # shape: (N,)
all_preds = np.argmax(all_probs, axis=1)

print(f'\nTotal test samples: {len(all_true)}')


In [ ]:
# --- Aggregate metrics ---
acc  = accuracy_score(all_true, all_preds)
prec = precision_score(all_true, all_preds, average='weighted', zero_division=0)
rec  = recall_score(all_true, all_preds, average='weighted', zero_division=0)
f1   = f1_score(all_true, all_preds, average='weighted', zero_division=0)

print('\n' + '=' * 55)
print('EVALUATION SUMMARY — MobileNetV2')
print('=' * 55)
print(f'  Accuracy  (weighted): {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision (weighted): {prec:.4f}')
print(f'  Recall    (weighted): {rec:.4f}')
print(f'  F1 Score  (weighted): {f1:.4f}')
print('=' * 55)

print('\nPer-class Classification Report:')
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=4))

# Save summary
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1 Score (weighted)'],
    'MobileNetV2': [acc, prec, rec, f1],
})
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_metrics.csv'), index=False)
print(f'\nMetrics saved to {OUTPUT_DIR}/evaluation_metrics.csv')


In [ ]:
# --- Confusion Matrix ---
def plot_confusion_matrix(true_labels, pred_labels, class_names, output_dir, model_name):
    cm_raw  = confusion_matrix(true_labels, pred_labels)
    cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, (matrix, title, fmt) in zip(axes, [
        (cm_raw,  'Count',          'd'),
        (cm_norm, 'Row-Normalized', '.2%'),
    ]):
        sns.heatmap(matrix, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
        ax.set_title(f'{model_name} — Confusion Matrix ({title})')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    fname = os.path.join(output_dir, 'confusion_matrix.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')


plot_confusion_matrix(all_true, all_preds, CLASS_NAMES, OUTPUT_DIR, 'MobileNetV2')


In [ ]:
# --- ROC Curves (One-vs-Rest, multi-class) ---
def plot_roc_curves(true_labels, pred_probs, class_names, output_dir, model_name):
    n = len(class_names)
    true_bin = label_binarize(true_labels, classes=list(range(n)))

    fig, ax = plt.subplots(figsize=(9, 7))
    colors = plt.cm.tab10(np.linspace(0, 1, n))
    all_fpr = np.linspace(0, 1, 200)
    mean_tpr = np.zeros(200)
    auc_scores = {}

    for i, (cls, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(true_bin[:, i], pred_probs[:, i])
        roc_auc = auc(fpr, tpr)
        auc_scores[cls] = roc_auc
        mean_tpr += np.interp(all_fpr, fpr, tpr)
        ax.plot(fpr, tpr, lw=2, color=color, label=f'{cls} (AUC={roc_auc:.4f})')

    mean_tpr /= n
    macro_auc = auc(all_fpr, mean_tpr)
    ax.plot(all_fpr, mean_tpr, 'k--', lw=2, label=f'Macro Avg (AUC={macro_auc:.4f})')
    ax.plot([0, 1], [0, 1], 'gray', ls=':', lw=1)
    ax.set(xlim=[0, 1], ylim=[0, 1.05],
           xlabel='False Positive Rate', ylabel='True Positive Rate',
           title=f'{model_name} — ROC Curves (One-vs-Rest)')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = os.path.join(output_dir, 'roc_curves.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Macro-average AUC: {macro_auc:.4f}')
    return auc_scores, macro_auc


auc_scores, macro_auc = plot_roc_curves(all_true, all_probs, CLASS_NAMES, OUTPUT_DIR, 'MobileNetV2')


## Section 12: Test-Time Augmentation (TTA)

TTA improves inference accuracy by averaging predictions over multiple augmented versions of each test image.

Reference: Krizhevsky et al. (2012). *ImageNet Classification with Deep Convolutional Neural Networks*. NeurIPS 2012.

In [ ]:
# TTA augmentation (lighter than training augmentation to avoid excessive distortion)
tta_augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(factor=0.10),
    layers.RandomBrightness(factor=0.1),
], name='tta_augmentation')


def predict_with_tta(model, dataset, n_passes=5):
    """
    Run inference with Test-Time Augmentation.

    Args:
        model: Trained Keras model.
        dataset: tf.data.Dataset (without augmentation).
        n_passes (int): Total passes per image (1 original + n_passes-1 augmented).

    Returns:
        avg_probs (np.ndarray): Averaged probabilities. Shape: (N, NUM_CLASSES).
        true_labels (np.ndarray): Ground-truth class indices. Shape: (N,).
    """
    avg_probs  = []
    true_labels = []

    for images, labels in dataset:
        batch_size = images.shape[0]
        batch_probs = np.zeros((batch_size, NUM_CLASSES), dtype=np.float32)

        # Original (no augmentation)
        batch_probs += model.predict(images, verbose=0)

        # Augmented passes
        for _ in range(n_passes - 1):
            aug_images = tta_augment(images, training=True)
            batch_probs += model.predict(aug_images, verbose=0)

        avg_probs.extend(batch_probs / n_passes)
        true_labels.extend(np.argmax(labels.numpy(), axis=1))

    return np.array(avg_probs), np.array(true_labels)


print('Running TTA (5 passes per image)...')
tta_probs, tta_true = predict_with_tta(best_model, test_ds, n_passes=5)
tta_preds = np.argmax(tta_probs, axis=1)

tta_acc = accuracy_score(tta_true, tta_preds)
tta_f1  = f1_score(tta_true, tta_preds, average='weighted', zero_division=0)

print(f'\nTTA Results:')
print(f'  Standard Accuracy: {acc:.4f}')
print(f'  TTA Accuracy:      {tta_acc:.4f}  (delta: {tta_acc-acc:+.4f})')
print(f'  Standard F1:       {f1:.4f}')
print(f'  TTA F1:            {tta_f1:.4f}  (delta: {tta_f1-f1:+.4f})')


## Section 13: Grad-CAM Visualization

**Grad-CAM** (Gradient-weighted Class Activation Mapping) highlights which regions of the input image most influenced the model's prediction.

This provides **visual explainability** — essential for agricultural AI to demonstrate that the model focuses on leaf discoloration regions, not background.

Reference: Selvaraju et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*. ICCV 2017. DOI:10.1109/ICCV.2017.74

In [ ]:
def make_gradcam_heatmap(grad_model, img_array):
    """
    Generate Grad-CAM heatmap.

    Args:
        grad_model: Model with outputs [last_conv_output, final_predictions].
        img_array: Input image array of shape (1, H, W, C), normalized to [-1,1].

    Returns:
        heatmap (np.ndarray): 2D heatmap, values in [0, 1].
        pred_class (int): Predicted class index.
        confidence (float): Softmax probability of predicted class.
    """
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        pred_class  = tf.argmax(predictions[0]).numpy()
        confidence  = float(predictions[0, pred_class])
        class_score = predictions[:, pred_class]

    # Gradient of the class score w.r.t. the last conv feature map
    grads = tape.gradient(class_score, conv_outputs)

    # Global average pool the gradients (channel-wise importance)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the feature maps by their gradient importance
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0)                          # ReLU
    heatmap = heatmap / (np.max(heatmap) + 1e-8)              # Normalize to [0,1]
    return heatmap, pred_class, confidence


def build_gradcam_model(full_model, base_model_name, last_conv_layer_name):
    """
    Build a model that outputs both the last conv layer activations and the final predictions.

    For MobileNetV2, the last meaningful conv activation before GAP is 'out_relu'
    (after the final 1x1 conv in the backbone).
    """
    backbone = full_model.get_layer(base_model_name)
    last_conv_output = backbone.get_layer(last_conv_layer_name).output

    # Sub-model: backbone inputs -> last conv output
    feature_extractor = keras.Model(
        inputs=backbone.inputs,
        outputs=last_conv_output,
        name='feature_extractor'
    )

    # Full grad model
    img_input = keras.Input(shape=INPUT_SHAPE, name='gradcam_input')
    features  = feature_extractor(img_input)
    preds     = full_model(img_input)

    return keras.Model(
        inputs=img_input,
        outputs=[features, preds],
        name='gradcam_model'
    )


# In MobileNetV2, 'out_relu' is the last ReLU6 activation before GlobalAveragePooling2D
# Verify: base_model.get_layer('out_relu') should not raise an error
LAST_CONV_NAME    = 'out_relu'
BASE_MODEL_NAME   = 'mobilenetv2_1.00_224'  # Default name from tf.keras.applications.MobileNetV2

try:
    gradcam_model = build_gradcam_model(best_model, BASE_MODEL_NAME, LAST_CONV_NAME)
    print(f'Grad-CAM model built. Target layer: {LAST_CONV_NAME}')
except ValueError as e:
    print(f'Error building Grad-CAM model: {e}')
    print('Available base model layers:')
    base = best_model.get_layer(BASE_MODEL_NAME)
    for l in base.layers[-10:]:
        print(f'  {l.name}')
    raise


In [ ]:
def display_gradcam_grid(gradcam_model, dataset, class_names, output_dir,
                          n_samples=4, model_name='MobileNetV2'):
    """Display Grad-CAM overlays for n_samples test images."""
    images_batch, labels_batch = next(iter(dataset))
    n = min(n_samples, len(images_batch))

    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3.8))
    if n == 1:
        axes = [axes]

    for i in range(n):
        img   = images_batch[i:i+1]       # (1, 224, 224, 3)
        label = labels_batch[i].numpy()
        true_cls = class_names[np.argmax(label)]

        heatmap, pred_class, confidence = make_gradcam_heatmap(gradcam_model, img)
        pred_cls = class_names[pred_class]

        # Denormalize image for display ([-1,1] -> [0,1])
        img_display = np.clip((img[0].numpy() + 1.0) / 2.0, 0, 1)

        # Resize heatmap to image size
        heatmap_resized = np.array(
            tf.image.resize(heatmap[..., np.newaxis], IMAGE_SIZE)
        ).squeeze()

        # Colorize heatmap and blend with original image
        heatmap_rgb = cm.jet(heatmap_resized)[..., :3]
        overlay     = np.clip(0.55 * img_display + 0.45 * heatmap_rgb, 0, 1)

        correct_marker = '\u2713' if pred_class == np.argmax(label) else '\u2717'

        axes[i][0].imshow(img_display)
        axes[i][0].set_title(f'Original\nTrue: {true_cls}', fontsize=10)
        axes[i][0].axis('off')

        axes[i][1].imshow(heatmap_resized, cmap='jet')
        axes[i][1].set_title('Grad-CAM Heatmap', fontsize=10)
        axes[i][1].axis('off')

        axes[i][2].imshow(overlay)
        axes[i][2].set_title(
            f'Overlay {correct_marker}\nPred: {pred_cls} ({confidence:.1%})', fontsize=10
        )
        axes[i][2].axis('off')

    plt.suptitle(f'{model_name} — Grad-CAM Explanations', fontsize=14)
    plt.tight_layout()
    fname = os.path.join(output_dir, 'gradcam_visualizations.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Grad-CAM saved: {fname}')


display_gradcam_grid(gradcam_model, test_ds, CLASS_NAMES, OUTPUT_DIR, n_samples=4)


## Section 14: Model Calibration

**Calibration** measures whether the model's confidence matches its actual accuracy.
A well-calibrated model predicting 80% confidence should be correct ~80% of the time.

**Expected Calibration Error (ECE):** Lower is better. Perfect calibration = ECE of 0.

Reference: Guo et al. (2017). *On Calibration of Modern Neural Networks*. ICML 2017.

In [ ]:
def compute_ece_and_plot(true_labels, pred_probs, predictions,
                          n_bins=10, output_dir=None, model_name='Model'):
    """
    Compute Expected Calibration Error and plot reliability diagram.

    Args:
        true_labels: Ground-truth class indices.
        pred_probs:  Softmax probability array (N, C).
        predictions: Predicted class indices.
        n_bins:      Number of confidence bins.
    """
    confidences = np.max(pred_probs, axis=1)
    correct     = (predictions == true_labels).astype(float)
    N           = len(true_labels)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_acc, bin_conf, bin_cnt = [], [], []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() > 0:
            bin_acc.append(correct[mask].mean())
            bin_conf.append(confidences[mask].mean())
            bin_cnt.append(mask.sum())
        else:
            bin_acc.append(0.0)
            bin_conf.append((lo + hi) / 2)
            bin_cnt.append(0)

    ece = sum(cnt / N * abs(a - c)
              for cnt, a, c in zip(bin_cnt, bin_acc, bin_conf))

    fig, ax = plt.subplots(figsize=(7, 7))
    width = 1.0 / n_bins * 0.85
    ax.bar(bin_conf, bin_acc, width=width, alpha=0.75, color='steelblue',
           label='Model accuracy per bin', zorder=3)
    ax.bar(bin_conf, [c - a for c, a in zip(bin_conf, bin_acc)],
           bottom=bin_acc, width=width, alpha=0.3, color='red',
           label='Gap (overconfidence)', zorder=3)
    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfect calibration')
    ax.set(xlabel='Confidence', ylabel='Accuracy',
           xlim=[0, 1], ylim=[0, 1],
           title=f'{model_name} — Reliability Diagram\nECE = {ece:.4f}')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, zorder=0)
    plt.tight_layout()

    if output_dir:
        fname = os.path.join(output_dir, 'calibration_reliability_diagram.png')
        plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Expected Calibration Error (ECE): {ece:.4f}')
    return ece


ece = compute_ece_and_plot(all_true, all_probs, all_preds, output_dir=OUTPUT_DIR,
                            model_name='MobileNetV2')


## Section 15: OOD Rejection Analysis

**Strategy (approved):** Confidence threshold as primary mechanism. A dedicated `Unknown` class is the recommended extension when non-Pechay training images are available.

**Limitations of confidence threshold:**
- Softmax outputs can be overconfident even for OOD inputs (Guo et al., 2017).
- A dedicated Unknown class in training data is strongly preferred.
- No method guarantees perfect rejection of all unknown images.

This section helps you tune `CONFIDENCE_THRESHOLD` based on your test set characteristics.

In [ ]:
confidences = np.max(all_probs, axis=1)
correct_mask = (all_preds == all_true)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# -- Confidence distribution --
axes[0].hist(confidences[correct_mask],  bins=25, alpha=0.7, color='green',
             density=True, label=f'Correct  ({correct_mask.sum()})')
axes[0].hist(confidences[~correct_mask], bins=25, alpha=0.7, color='red',
             density=True, label=f'Incorrect ({(~correct_mask).sum()})')
axes[0].axvline(CONFIDENCE_THRESHOLD, color='black', ls='--', lw=2,
                label=f'Threshold = {CONFIDENCE_THRESHOLD}')
axes[0].set(xlabel='Confidence (max softmax)', ylabel='Density',
             title='Confidence Distribution on Test Set')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# -- Accuracy-Coverage trade-off across thresholds --
thresholds = np.arange(0.05, 1.0, 0.05)
coverages, accs_at_thresh = [], []
for t in thresholds:
    mask = confidences >= t
    coverages.append(mask.mean())
    if mask.sum() > 0:
        accs_at_thresh.append(accuracy_score(all_true[mask], all_preds[mask]))
    else:
        accs_at_thresh.append(1.0)

axes[1].plot(coverages, accs_at_thresh, 'b-o', markersize=5, lw=2)
axes[1].axvline((confidences >= CONFIDENCE_THRESHOLD).mean(),
                color='red', ls='--', lw=1.5,
                label=f'Threshold={CONFIDENCE_THRESHOLD}')
axes[1].set(xlabel='Coverage (fraction of accepted)', ylabel='Accuracy on Accepted',
             title='Accuracy-Coverage Trade-off')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('MobileNetV2 — OOD Rejection Analysis', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ood_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

accepted = confidences >= CONFIDENCE_THRESHOLD
print(f'\nOOD Threshold = {CONFIDENCE_THRESHOLD}')
print(f'  Accepted: {accepted.sum()}/{len(confidences)} ({accepted.mean():.1%})')
print(f'  Rejected: {(~accepted).sum()}/{len(confidences)} ({(~accepted).mean():.1%})')
if accepted.sum() > 0:
    print(f'  Accuracy on accepted: {accuracy_score(all_true[accepted], all_preds[accepted]):.4f}')

# Entropy-based rejection (complementary metric)
epsilon = 1e-9
entropy = -np.sum(all_probs * np.log(all_probs + epsilon), axis=1)
max_entropy = np.log(NUM_CLASSES)
norm_entropy = entropy / max_entropy
print(f'\nEntropy stats (normalized 0=certain, 1=max uncertainty):')
print(f'  Mean: {norm_entropy.mean():.4f}  |  Std: {norm_entropy.std():.4f}')
print(f'  Correct predictions — mean entropy: {norm_entropy[correct_mask].mean():.4f}')
print(f'  Incorrect predictions — mean entropy: {norm_entropy[~correct_mask].mean():.4f}')


## Section 16: TFLite Conversion

Three TFLite formats are produced:

| Format | Size Reduction | Accuracy | Recommended Use |
|--------|---------------|----------|-----------------|
| Float32 | None (baseline) | Highest | Development/debugging |
| Float16 | ~2x smaller | Near-identical (GPU) | Android GPU delegate |
| Int8 | ~4x smaller | Minor loss | Android CPU (fastest) |

**Recommendation for thesis Android app:** Use **Float16** for the primary deployed model (good balance of size and accuracy). Int8 as optional CPU-only fallback.

In [ ]:
# --- Save as SavedModel (required input for TFLite converter) ---
best_model.save(SAVEDMODEL_PATH)
print(f'SavedModel saved: {SAVEDMODEL_PATH}')


In [ ]:
def convert_tflite(savedmodel_path, output_path, quantization=None,
                   representative_data_fn=None):
    """
    Convert a SavedModel to TFLite format.

    Args:
        savedmodel_path (str): Path to SavedModel directory.
        output_path (str): Output .tflite file path.
        quantization (str or None): None | 'float16' | 'int8'.
        representative_data_fn (callable): Generator for int8 calibration (optional).

    Returns:
        float: Model file size in KB.
    """
    converter = tf.lite.TFLiteConverter.from_saved_model(savedmodel_path)

    if quantization == 'float16':
        # Reduce model size by ~50%; computations in float16 on GPU delegate
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quantization == 'int8':
        # Dynamic range quantization by default; full int8 if representative data provided
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        if representative_data_fn:
            converter.representative_dataset = representative_data_fn
            # Keep float32 I/O for Android compatibility (avoids dequantize step)
            converter.inference_input_type  = tf.float32
            converter.inference_output_type = tf.float32

    tflite_model = converter.convert()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'wb') as f:
        f.write(tflite_model)

    size_kb = os.path.getsize(output_path) / 1024
    print(f'  \u2705 {os.path.basename(output_path)}: {size_kb:.1f} KB')
    return size_kb


def representative_dataset_gen():
    """Yield 100 representative training samples for int8 calibration."""
    count = 0
    for images, _ in train_ds:
        for img in images:
            if count >= 100:
                return
            yield [tf.expand_dims(img, 0)]
            count += 1


print('Converting to TFLite...')
size_f32  = convert_tflite(SAVEDMODEL_PATH, TFLITE_PATH_F32,  quantization=None)
size_f16  = convert_tflite(SAVEDMODEL_PATH, TFLITE_PATH_F16,  quantization='float16')
size_int8 = convert_tflite(SAVEDMODEL_PATH, TFLITE_PATH_INT8, quantization='int8',
                            representative_data_fn=representative_dataset_gen)

print(f'\nSize summary:')
print(f'  Float32:  {size_f32:.1f} KB  (baseline)')
print(f'  Float16:  {size_f16:.1f} KB  ({size_f32/size_f16:.1f}x smaller)')
print(f'  Int8:     {size_int8:.1f} KB  ({size_f32/size_int8:.1f}x smaller)')


In [ ]:
def verify_tflite(tflite_path, test_dataset, class_names, label):
    """
    Run a quick inference check on the TFLite model.
    Computes accuracy over the full test set.
    """
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    inp_det = interpreter.get_input_details()[0]
    out_det = interpreter.get_output_details()[0]

    all_tflite_preds, all_tflite_true = [], []
    for images, labels in test_dataset:
        for img, lbl in zip(images.numpy(), labels.numpy()):
            inp = np.expand_dims(img.astype(inp_det['dtype']), 0)
            interpreter.set_tensor(inp_det['index'], inp)
            interpreter.invoke()
            output = interpreter.get_tensor(out_det['index'])[0]
            all_tflite_preds.append(np.argmax(output))
            all_tflite_true.append(np.argmax(lbl))

    tflite_acc = accuracy_score(all_tflite_true, all_tflite_preds)
    print(f'  {label}: input_dtype={inp_det["dtype"].__name__}, '
          f'input_shape={inp_det["shape"].tolist()}, '
          f'test_accuracy={tflite_acc:.4f}')
    return tflite_acc


print('\nVerifying TFLite models on test set...')
acc_f32  = verify_tflite(TFLITE_PATH_F32,  test_ds, CLASS_NAMES, 'Float32')
acc_f16  = verify_tflite(TFLITE_PATH_F16,  test_ds, CLASS_NAMES, 'Float16')
acc_int8 = verify_tflite(TFLITE_PATH_INT8, test_ds, CLASS_NAMES, 'Int8')

print(f'\nAccuracy drop from Float32 baseline:')
print(f'  Float16 drop: {(acc_f32 - acc_f16)*100:+.2f}%')
print(f'  Int8 drop:    {(acc_f32 - acc_int8)*100:+.2f}%')


## Section 17: Final Summary Report

In [ ]:
print('\n' + '=' * 65)
print('MOBILENETV2 — FINAL EVALUATION REPORT')
print('=' * 65)
print(f'  Architecture:       MobileNetV2 (alpha=1.0, ImageNet pretrained)')
print(f'  Input size:         {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}x{CHANNELS}')
print(f'  Classes ({NUM_CLASSES}):       {CLASS_NAMES}')
print(f'  Training phases:    3 (Head | Top Fine-Tune | Full Fine-Tune)')
print(f'  Augmentation:       Flip, Rotation\u00b115\u00b0, Zoom\u00b110%, Brightness\u00b120%, Contrast\u00b110%')
print(f'  Label smoothing:    {LABEL_SMOOTHING}')
print(f'  Dropout:            {DROPOUT_RATE}')
print(f'  L2 regularization:  {L2_REGULARIZATION}')
print(f'  OOD threshold:      {CONFIDENCE_THRESHOLD}')
print('-' * 65)
print('  TEST SET PERFORMANCE')
print(f'    Accuracy  (standard): {acc:.4f}  ({acc*100:.2f}%)')
print(f'    Accuracy  (TTA):      {tta_acc:.4f}  ({tta_acc*100:.2f}%)')
print(f'    Precision (weighted): {prec:.4f}')
print(f'    Recall    (weighted): {rec:.4f}')
print(f'    F1 Score  (weighted): {f1:.4f}')
print(f'    Macro AUC:            {macro_auc:.4f}')
print(f'    Calibration (ECE):    {ece:.4f}')
print('-' * 65)
print('  TFLITE MODELS')
print(f'    Float32:  {size_f32:.1f} KB   accuracy: {acc_f32:.4f}')
print(f'    Float16:  {size_f16:.1f} KB   accuracy: {acc_f16:.4f}  (recommended for Android)')
print(f'    Int8:     {size_int8:.1f} KB   accuracy: {acc_int8:.4f}')
print('-' * 65)
print('  OUTPUT FILES')
for f in sorted(Path(OUTPUT_DIR).rglob('*')):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f'    {f.name:<45} {size:>8.1f} KB')
print('=' * 65)
print('\u2705 MobileNetV2 notebook complete. Proceed to MobileNetV3 notebook.')
